# P1 포즈 모델 비교 — Colab GPU

순서대로 실행하거나 **런타임 → 모두 실행**을 누른다.

- 시작 전: **런타임 → 런타임 유형 변경 → GPU**(A100 > L4 > T4 순으로 좋음)
- 2번 칸에서 Google Drive 연결을 허용한다.
- 끊겨도 다시 '모두 실행'하면 끝난 구간은 건너뛴다.
- 결과: 내 드라이브 `AI-sfa/02_분석자료/P1_포즈비교/` — Claude가 Drive에서 직접 읽는다(첨부 필요 없음).
- 표본: 애니메이터 5명 × 영상 4개 × 연속 32프레임 2구간 = 1280프레임

In [ ]:
# 1. 준비(3~5분)
!pip -q install -U "transformers>=5.20" accelerate ultralytics rtmlib onnxruntime-gpu
import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

In [ ]:
# 2. 내 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 설정
import os, json, shutil, time, traceback
import cv2, numpy as np, torch
from PIL import Image
ROOT = '/content/drive/MyDrive/AI-sfa'
OUT = ROOT + '/02_분석자료/P1_포즈비교'
os.makedirs(OUT, exist_ok=True)
CLIPS = json.loads(r'''[{"animator": "nian41", "id": "216073", "path": "01_원본영상/nian41/216073_f63b8906f518f9f4322928c5441f48f1.mp4", "start": 60, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "nian41", "id": "216073", "path": "01_원본영상/nian41/216073_f63b8906f518f9f4322928c5441f48f1.mp4", "start": 131, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "nian41", "id": "280556", "path": "01_원본영상/nian41/280556_5db093050985f3330778698b999a1df8.mp4", "start": 1263, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "nian41", "id": "280556", "path": "01_원본영상/nian41/280556_5db093050985f3330778698b999a1df8.mp4", "start": 2737, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "nian41", "id": "301142", "path": "01_원본영상/nian41/301142_95bd08a7d77433834c6d677b7d59764b.mp4", "start": 576, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "nian41", "id": "301142", "path": "01_원본영상/nian41/301142_95bd08a7d77433834c6d677b7d59764b.mp4", "start": 1248, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "nian41", "id": "280534", "path": "01_원본영상/nian41/280534_427f5038c2eb53f02c10e2985526c09c.mp4", "start": 254, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "nian41", "id": "280534", "path": "01_원본영상/nian41/280534_427f5038c2eb53f02c10e2985526c09c.mp4", "start": 551, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "yutaka_nakamura", "id": "224077", "path": "01_원본영상/yutaka_nakamura_디지털/224077_2017_완성.mp4", "start": 126, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "yutaka_nakamura", "id": "224077", "path": "01_원본영상/yutaka_nakamura_디지털/224077_2017_완성.mp4", "start": 273, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "yutaka_nakamura", "id": "69067", "path": "01_원본영상/yutaka_nakamura_디지털/69067_2004_완성.mp4", "start": 40, "count": 32, "fps": "24000/1001", "w": 640, "h": 480}, {"animator": "yutaka_nakamura", "id": "69067", "path": "01_원본영상/yutaka_nakamura_디지털/69067_2004_완성.mp4", "start": 87, "count": 32, "fps": "24000/1001", "w": 640, "h": 480}, {"animator": "yutaka_nakamura", "id": "41027", "path": "01_원본영상/yutaka_nakamura_디지털/41027_2017_원화.mp4", "start": 36, "count": 32, "fps": "24/1", "w": 860, "h": 480}, {"animator": "yutaka_nakamura", "id": "41027", "path": "01_원본영상/yutaka_nakamura_디지털/41027_2017_원화.mp4", "start": 78, "count": 32, "fps": "24/1", "w": 860, "h": 480}, {"animator": "yutaka_nakamura", "id": "196315", "path": "01_원본영상/yutaka_nakamura_디지털/196315_2021_완성.mp4", "start": 219, "count": 32, "fps": "24000/1001", "w": 1146, "h": 480}, {"animator": "yutaka_nakamura", "id": "196315", "path": "01_원본영상/yutaka_nakamura_디지털/196315_2021_완성.mp4", "start": 475, "count": 32, "fps": "24000/1001", "w": 1146, "h": 480}, {"animator": "optical-core", "id": "47588", "path": "01_원본영상/optical-core/47588_ca61835e7a2762b3b27ca05f1b458f45.mp4", "start": 145, "count": 32, "fps": "25/1", "w": 400, "h": 224}, {"animator": "optical-core", "id": "47588", "path": "01_원본영상/optical-core/47588_ca61835e7a2762b3b27ca05f1b458f45.mp4", "start": 315, "count": 32, "fps": "25/1", "w": 400, "h": 224}, {"animator": "optical-core", "id": "86099", "path": "01_원본영상/optical-core/presumed/86099_663db98332eb9d1389d888dcdcb87089.mp4", "start": 63, "count": 32, "fps": "30/1", "w": 1032, "h": 480}, {"animator": "optical-core", "id": "86099", "path": "01_원본영상/optical-core/presumed/86099_663db98332eb9d1389d888dcdcb87089.mp4", "start": 138, "count": 32, "fps": "30/1", "w": 1032, "h": 480}, {"animator": "optical-core", "id": "32636", "path": "01_원본영상/optical-core/32636_88aeeafce36103ddfecdd567f6a3cc35.mp4", "start": 239, "count": 32, "fps": "37/2", "w": 854, "h": 480}, {"animator": "optical-core", "id": "32636", "path": "01_원본영상/optical-core/32636_88aeeafce36103ddfecdd567f6a3cc35.mp4", "start": 518, "count": 32, "fps": "37/2", "w": 854, "h": 480}, {"animator": "optical-core", "id": "284200", "path": "01_원본영상/optical-core/284200_d6730c1693258fd80ed37c814c90f7e1.mp4", "start": 461, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "optical-core", "id": "284200", "path": "01_원본영상/optical-core/284200_d6730c1693258fd80ed37c814c90f7e1.mp4", "start": 999, "count": 32, "fps": "24/1", "w": 854, "h": 480}, {"animator": "riooo", "id": "112572", "path": "01_원본영상/riooo/112572_57039fe98f8c1d36fe0787f7456b80cb.mp4", "start": 163, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "riooo", "id": "112572", "path": "01_원본영상/riooo/112572_57039fe98f8c1d36fe0787f7456b80cb.mp4", "start": 354, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "riooo", "id": "209782", "path": "01_원본영상/riooo/209782_f34e60c11f609c7179f85712a504ecdd.mp4", "start": 189, "count": 32, "fps": "24/1", "w": 1132, "h": 480}, {"animator": "riooo", "id": "209782", "path": "01_원본영상/riooo/209782_f34e60c11f609c7179f85712a504ecdd.mp4", "start": 410, "count": 32, "fps": "24/1", "w": 1132, "h": 480}, {"animator": "riooo", "id": "230407", "path": "01_원본영상/riooo/230407_50b6884c679d0672e3bbe8b4aa68cd68.mp4", "start": 122, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "riooo", "id": "230407", "path": "01_원본영상/riooo/230407_50b6884c679d0672e3bbe8b4aa68cd68.mp4", "start": 265, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "riooo", "id": "128305", "path": "01_원본영상/riooo/128305_b1a4edf0bf7a8ca9503be736331ca824.mp4", "start": 203, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "riooo", "id": "128305", "path": "01_원본영상/riooo/128305_b1a4edf0bf7a8ca9503be736331ca824.mp4", "start": 441, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "yukina_kosaka", "id": "200394", "path": "01_원본영상/yukina_kosaka/presumed/200394_a1d223d17f7b834c96781fbd2f1d058a.mp4", "start": 36, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "yukina_kosaka", "id": "200394", "path": "01_원본영상/yukina_kosaka/presumed/200394_a1d223d17f7b834c96781fbd2f1d058a.mp4", "start": 78, "count": 32, "fps": "24000/1001", "w": 852, "h": 480}, {"animator": "yukina_kosaka", "id": "316879", "path": "01_원본영상/yukina_kosaka/presumed/316879_cb6ebbbb371247c5945d6487eb93bbaf.mp4", "start": 180, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "yukina_kosaka", "id": "316879", "path": "01_원본영상/yukina_kosaka/presumed/316879_cb6ebbbb371247c5945d6487eb93bbaf.mp4", "start": 391, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "yukina_kosaka", "id": "174523", "path": "01_원본영상/yukina_kosaka/174523_fef0864c32df138adf4776000c59669e.mp4", "start": 123, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "yukina_kosaka", "id": "174523", "path": "01_원본영상/yukina_kosaka/174523_fef0864c32df138adf4776000c59669e.mp4", "start": 267, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "yukina_kosaka", "id": "236403", "path": "01_원본영상/yukina_kosaka/presumed/236403_7a81433c8099ee659f729de267592182.mp4", "start": 536, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}, {"animator": "yukina_kosaka", "id": "236403", "path": "01_원본영상/yukina_kosaka/presumed/236403_7a81433c8099ee659f729de267592182.mp4", "start": 1161, "count": 32, "fps": "24000/1001", "w": 854, "h": 480}]''')
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
BIG = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 30e9
DT = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
print(len(CLIPS), '구간', GPU, '큰 모델(5B):', BIG, DT)

def key(c): return f"{c['animator']}_{c['id']}_{c['start']}"

_cache = {}
def frames(c):
    k = key(c)
    if k in _cache: return _cache[k]
    src = ROOT + '/' + c['path']; loc = '/content/v_' + os.path.basename(src)
    if not os.path.exists(loc): shutil.copy(src, loc)
    cap = cv2.VideoCapture(loc); fs = []; i = 0
    while len(fs) < c['count']:
        ok, im = cap.read()
        if not ok: break
        if i >= c['start']: fs.append(im)
        i += 1
    cap.release(); _cache.clear(); _cache[k] = fs
    return fs

def save(model, c, rows, labels=None):
    d = f'{OUT}/{model}'; os.makedirs(d, exist_ok=True)
    if labels is not None and not os.path.exists(d + '/_labels.json'):
        json.dump(labels, open(d + '/_labels.json', 'w'))
    json.dump({'clip': c, 'frames': rows}, open(f'{d}/{key(c)}.json', 'w'))

def done(model, c): return os.path.exists(f'{OUT}/{model}/{key(c)}.json')

def r1(a): return np.round(np.asarray(a, dtype=float), 1).tolist()
def kp3(a):  # x, y는 소수 1자리, 점수는 소수 3자리
    a = np.asarray(a, dtype=float); return np.concatenate([a[:, :2].round(1), a[:, 2:3].round(3)], 1).tolist()

In [ ]:
# 4. 사람 상자: rtmlib의 HumanArt 검출기(그림·애니 포함 학습, 기준값 0.3).
#    모든 top-down 모델(RTMW·ViTPose·Sapiens2)이 같은 상자를 쓴다. 상자가 없으면 화면 전체.
from rtmlib import Wholebody
WB = Wholebody(mode='performance', backend='onnxruntime', device='cuda')
WB.det_model.score_thr = 0.3
BOXES = {}
for c in CLIPS:
    p = f'{OUT}/_boxes/{key(c)}.json'
    if os.path.exists(p): BOXES[key(c)] = json.load(open(p)); continue
    fs = frames(c); h, w = fs[0].shape[:2]; bl = []
    for im in fs:
        b = [[x1, y1, x2 - x1, y2 - y1] for x1, y1, x2, y2 in np.asarray(WB.det_model(im)).reshape(-1, 4).tolist()]
        bl.append(r1(b) if b else [[0, 0, w, h]])
    os.makedirs(f'{OUT}/_boxes', exist_ok=True); json.dump(bl, open(p, 'w')); BOXES[key(c)] = bl
print('상자 완료', len(BOXES), '구간')

In [ ]:
# 5. 모델 A: YOLO11x-pose (상자도 스스로 찾음)
def run_yolo():
    from ultralytics import YOLO
    m = YOLO('yolo11x-pose.pt')
    for c in CLIPS:
        if done('yolo11x', c): continue
        rows = []
        for im in frames(c):
            r = m.predict(im, conf=0.10, imgsz=960, verbose=False)[0]
            ppl = []
            if r.keypoints is not None and len(r.boxes):
                for b, k in zip(r.boxes.xyxy.cpu().numpy(), r.keypoints.data.cpu().numpy()):
                    ppl.append({'box': r1([b[0], b[1], b[2] - b[0], b[3] - b[1]]), 'kp': kp3(k)})
            rows.append(ppl)
        save('yolo11x', c, rows, 'coco17')
    print('yolo11x 완료')
try: run_yolo()
except Exception: traceback.print_exc()

In [ ]:
# 6. 모델 B: RTMW-x 전신 133점(rtmlib, 4번 상자)
def run_rtmw():
    for c in CLIPS:
        if done('rtmw_x', c): continue
        rows = []
        for im, bx in zip(frames(c), BOXES[key(c)]):
            xyxy = np.array([[x, y, x + w, y + h] for x, y, w, h in bx])
            kps, sc = WB.pose_model(im, bboxes=xyxy)
            rows.append([{'box': b, 'kp': kp3(np.concatenate([k, s[:, None]], 1))} for b, k, s in zip(bx, kps, sc)])
        save('rtmw_x', c, rows, 'coco_wholebody133')
    print('rtmw_x 완료')
try: run_rtmw()
except Exception: traceback.print_exc()

In [ ]:
# 7. 모델 C·D: ViTPose+-H, Sapiens2 (4번 상자 사용, top-down)
from transformers import AutoProcessor, VitPoseForPoseEstimation, Sapiens2ForPoseEstimation

def run_topdown(name, repo, cls, vit=False):
    proc = AutoProcessor.from_pretrained(repo)
    m = cls.from_pretrained(repo, dtype=DT).cuda().eval()
    labels = [m.config.id2label[i] for i in range(len(m.config.id2label))] if m.config.id2label else None
    for c in CLIPS:
        if done(name, c): continue
        rows = []
        for im, bx in zip(frames(c), BOXES[key(c)]):
            pil = Image.fromarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            inp = proc(images=[pil], boxes=[bx], return_tensors='pt').to('cuda')
            inp['pixel_values'] = inp['pixel_values'].to(DT)
            with torch.no_grad():
                if vit:
                    out = m(**inp, dataset_index=torch.zeros(len(bx), dtype=torch.long, device='cuda'))
                else:
                    out = m(pixel_values=inp['pixel_values'])
            out.heatmaps = out.heatmaps.float()
            res = proc.post_process_pose_estimation(out, boxes=[bx])[0]
            rows.append([{'box': b, 'kp': kp3(torch.cat([p['keypoints'].cpu(), p['scores'].cpu()[:, None]], 1))}
                         for b, p in zip(bx, res)])
        save(name, c, rows, labels)
    del m; torch.cuda.empty_cache(); print(name, '완료')

for name, repo, cls, vit, need_big in [
    ('vitpose_plus_h', 'usyd-community/vitpose-plus-huge', VitPoseForPoseEstimation, True, False),
    ('sapiens2_1b', 'facebook/sapiens2-pose-1b', Sapiens2ForPoseEstimation, False, False),
    ('sapiens2_5b', 'facebook/sapiens2-pose-5b', Sapiens2ForPoseEstimation, False, True),
]:
    if need_big and not BIG: print(name, '건너뜀(GPU 메모리 30GB 미만)'); continue
    t = time.time()
    try: run_topdown(name, repo, cls, vit)
    except Exception: traceback.print_exc()
    print(name, round(time.time() - t), '초')

In [ ]:
# 8. 확인: 모델별 저장 구간 수
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f'{OUT}/{d}'): print(d, len([x for x in os.listdir(f'{OUT}/{d}') if not x.startswith('_')]))
open(OUT + '/_gpu.txt', 'w').write(GPU)